# 01 - Exploring the MultiModal PlantMetWiki graph

## Research context and motivation

PlantMetWiki is a cross-species metabolic knowledge graph built from WikiPathways, covering
**424 organisms** (plants, fungi, bacteria, and animals) across **2,478 pathways**.
The graph encodes **84,470 biochemical reactions** (Interaction/Conversion nodes) — but only
**4,572 of them (5.4%) have a known catalysing enzyme** assigned. The remaining
**79,898 reactions (94.6%) are 'orphan reactions'**: the chemistry is documented, but no
gene or protein has been linked to it.

On the protein side the picture is equally sparse: **13,682 proteins** are in the graph,
but only **3,456 (25.3%) are annotated as catalysing any reaction**. Of all proteins,
**38.3% lack an ESM sequence embedding** (protein characterisation is incomplete even where
sequences exist).

This gap is the core biological problem this project addresses:

> **Given a metabolic network with known reactions but sparse enzyme assignments, which
> proteins catalyse which reactions?**

This is the **enzyme discovery** or **orphan-reaction assignment** problem. It is framed as
link prediction in the **Conversion → Protein direction**: for a given reaction (Conversion
node), rank all candidate proteins and identify which one catalyses it. The alternative
framing — given a protein, predict which reactions it catalyses (Protein → Conversion) — is
the protein *annotation* problem, which is a different task and less relevant here because
our metabolic networks are reaction-rich but enzyme-sparse: we know *what happens*, we just
do not know *who does it*.

The model must generalise **across species** (taxa split): the val/test organisms are held out
entirely from training, so the model cannot rely on having seen those proteins before. It must
use protein sequence (ESM embeddings) and reaction chemistry (EC class features, substrate/
product metabolite graph) to transfer enzyme–reaction knowledge from characterised to
under-characterised organisms.

---

This notebook explores the processed PlantMetWiki graph after RDF parsing and
conversion to PyTorch Geometric `HeteroData`.

It loads:
- `data/processed/nodes.tsv`
- `data/processed/edges.tsv`
- `data/processed/heterodata.pt`

and produces:
- schema summaries and node/edge count tables
- simple graph statistics and annotation-coverage diagnostics
- reproducible figures in `reports/figures/`


In [ ]:
from pathlib import Path
import sys, platform, random

import pandas as pd
import torch
import matplotlib.pyplot as plt

try:
    import torch_geometric
except ImportError:
    torch_geometric = None

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 120

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR  = ROOT / "data" / "processed"
FIG_DIR   = ROOT / "reports" / "figures"
TABLE_DIR = ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

NODES_PATH  = DATA_DIR / "nodes.tsv"
EDGES_PATH  = DATA_DIR / "edges.tsv"
HETERO_PATH = DATA_DIR / "heterodata.pt"

print(f"Python {sys.version.split()[0]} | Torch {torch.__version__} | PyG {getattr(torch_geometric, '__version__', 'n/a')}")
print(f"ROOT: {ROOT}")
for p in [NODES_PATH, EDGES_PATH, HETERO_PATH]:
    status = "OK" if p.exists() else "MISSING – run pipeline scripts"
    print(f"  {p.name}: {status}")

nodes = pd.read_csv(NODES_PATH, sep="\t")
edges = pd.read_csv(EDGES_PATH, sep="\t")

if HETERO_PATH.exists():
    data = torch.load(HETERO_PATH, weights_only=False)
    print("\n", data)
else:
    data = None
    print("\nheterodata.pt not found. Run:  python scripts/build_pyg.py")

## 1. Inspect the Table Schema

In [ ]:
print("Node columns:", nodes.columns.tolist())
print("Edge columns:", edges.columns.tolist())
print()
display(nodes.head(3))
display(edges.head(3))

## 2. Node type distribution

In [ ]:
node_counts = (
    nodes["node_type"].value_counts()
    .rename_axis("node_type").reset_index(name="count")
)
print(node_counts.to_string(index=False))
node_counts.to_csv(TABLE_DIR / "node_type_counts.tsv", sep="\t", index=False)

fig, ax = plt.subplots()
ax.barh(node_counts["node_type"], node_counts["count"])
ax.set_xlabel("Count")
ax.set_title("Node type distribution")
plt.tight_layout()
fig.savefig(FIG_DIR / "node_type_distribution.png")
plt.show()

## 3. Edge type distribution

In [ ]:
edge_counts = (
    edges.groupby(["src_type", "rel", "dst_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print(edge_counts.to_string(index=False))
edge_counts.to_csv(TABLE_DIR / "edge_type_counts.tsv", sep="\t", index=False)

top = edge_counts.head(15).copy()
top["label"] = top["src_type"] + " -[" + top["rel"] + "]-> " + top["dst_type"]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top["label"][::-1], top["count"][::-1])
ax.set_xlabel("Count")
ax.set_title("Top 15 edge types")
plt.tight_layout()
fig.savefig(FIG_DIR / "edge_type_distribution.png")
plt.show()

### 3.1 What each edge type means — and the `is_about` edge explained

The 24 edge types group into four semantic categories:

---

**Group 1 — Task edges**

| Relation | Meaning |
|----------|--------|
| `catalyzes` | Target: enzyme (Protein) catalyses a reaction (Interaction/Conversion). The link-prediction objective. |
| `catalyzed_by` | Reverse of `catalyzes`. Only training-positive edges enter the MP graph; val/test never included. **Removed by default**, because it creates a 2-hop P→C→P training shortcut that lets the model memorise training pairs. |

**Group 2 — Reaction structure / chemistry**

| Relation | Meaning |
|----------|--------|
| `participants` | Metabolite (or sub-Interaction) takes part in a reaction. Main metabolite→reaction connectivity. |
| `source` / `target` | Directed role: reactant vs. product. |
| `is_part_of` (→ Interaction) | Metabolite or sub-reaction is a component of a reaction. Legitimate chemistry — do NOT remove. |
| `is_part_of` (→ Pathway) | Entity belongs to a metabolic pathway. **Removed by default** (`remove_is_part_of=True`): pathway co-membership inflates AUC to ≥0.98 without biological learning. |

**Group 3 — Gene / organism structure**

| Relation | Meaning |
|----------|--------|
| `encodes` / `encoded_by` | Gene (GeneProduct) → Protein. Derived from `TranscriptionTranslation` nodes. |
| `organism` | Maps a protein/gene/metabolite to its plant species (NCBITaxon). Used to define the taxa split. |

**Group 4 — `is_about`: WikiPathways entity identity links**

`is_about` is easy to misread, so its meaning is spelled out here.

`is_about` comes from the WikiPathways RDF predicate `wp:isAbout` (line 71 of
`scripts/rdf_to_typed_tables.py`). It is **not** a similarity, homology, or shortcut
relationship. It is a **provenance / identity link** built into WikiPathways' own GPML data model.

**Background:** In WikiPathways, every entity in a pathway diagram has two URIs:
1. A **canonical external ID** — e.g. `https://identifiers.org/uniprot/Q7G9P4`
2. A **pathway-local DataNode** — e.g. `.../Pathway/PC1/DataNode/AT2G27150_MONOMER`

`wp:isAbout` states: *"this canonical entity (Q7G9P4) is the real-world object depicted by
this DataNode (AT2G27150_MONOMER) in pathway PC1."*

Once DataNodes are reclassified into typed nodes, both the canonical ID and
the DataNode alias have the same node type — producing `is_about` edges between same-typed nodes.

**What each `is_about` type actually connects (verified by graph inspection):**

| Edge type | Count | What it connects | A shortcut? |
|-----------|------:|-----------------|-------------|
| `(Interaction, is_about, Interaction)` | 29,740 | ALL connect the **same canonical reaction** under different pathway URIs for different organisms: `RXN_18780_in_PC1 → RXN_18780_in_PC3`. Same reaction ID, different node indices because it appears in multiple organisms' pathway files. | **No** — this is the **only** structural bridge between cross-organism instances of the same reaction. |
| `(Protein, is_about, Protein)` | 9,932 | UniProt canonical protein → PlantCyc DataNode alias (e.g. `Q7G9P4 → AT2G27150_MONOMER`). Same protein, two database IDs. | **No** — DataNode destinations have **zero** `catalyzes` edges; no label leakage. |
| `(Metabolite, is_about, Metabolite)` | 12,812 | Same compound (InChIKey) → DataNode alias. | Identity, not similarity. |
| `(GeneProduct, is_about, GeneProduct)` | 4,580 | Same gene product, different pathway URIs. | Identity. |
| `(Pathway, is_about, Pathway)` | 2,478 | Pathway diagram instance → canonical pathway record. | Identity. |

**Why `is_about` edges are load-bearing for cross-species generalisation:**

The `(Interaction, is_about, Interaction)` edges are the **only** structural connection between
different organisms' versions of the same biochemical reaction. Under the taxa (organism-holdout)
split:
- A training protein from *A. thaliana* catalyses `RXN_18780` → represented as node A in the graph
- A held-out test protein from *G. max* catalyses `RXN_18780` → represented as node B
- Nodes A and B are different PyG node indices. Without `is_about`, they are **completely
  disconnected** — the GNN cannot know they are the same reaction.
- With `is_about` (A ↔ B), the GNN learns a shared reaction representation across organisms.

**Ablation evidence:** removing all `is_about` edges makes training fail. AUC inverts to 0.188 by epoch 10 and test P-H@50 drops to 0.003, below the random baseline of 0.59%, because training degrades the pre-trained ESM-C embeddings instead of improving them.

**Conclusion:** Treat `is_about` edges like `participants` and `source/target` —
load-bearing structural edges from the WikiPathways data model. Do not remove them.


## 4. Degree statistics per node type

In [ ]:
all_endpoints = pd.concat([
    edges[["src", "src_type"]].rename(columns={"src": "node_id", "src_type": "node_type"}),
    edges[["dst", "dst_type"]].rename(columns={"dst": "node_id", "dst_type": "node_type"}),
])
degrees = all_endpoints.groupby(["node_type", "node_id"]).size().reset_index(name="degree")

degree_stats = (
    degrees.groupby("node_type")["degree"]
    .describe(percentiles=[0.5, 0.75, 0.9, 0.95])
    .round(1)
)
display(degree_stats)
degree_stats.to_csv(TABLE_DIR / "degree_stats.tsv", sep="\t")

## 5. Graph composition, structure & organism (taxonomy) coverage

In [ ]:
from IPython.display import Markdown

# ── Graph composition ──────────────────────────────────────────────────────────
def safe_count(node_type):
    """0 if a node type is not present (e.g. DataNode/External, which Section 9 removes)."""
    m = node_counts.loc[node_counts["node_type"] == node_type, "count"]
    return int(m.values[0]) if len(m) else 0


total_nodes  = node_counts["count"].sum()
n_interaction = safe_count("Interaction")
n_metabolite  = safe_count("Metabolite")
n_protein     = safe_count("Protein")
n_geneproduct = safe_count("GeneProduct")
n_datanode    = safe_count("DataNode")
n_pathway     = safe_count("Pathway")
n_organism    = safe_count("Organism")
n_external_nodes = safe_count("External")
pct_interaction  = 100 * n_interaction / total_nodes

# ── Edge structure ─────────────────────────────────────────────────────────────
total_edges        = int(edge_counts["count"].sum())
n_edge_types       = len(edge_counts)
directional_edges  = int(edge_counts[edge_counts["rel"].isin(["source", "target"])]["count"].sum())
participants_edges = int(edge_counts[edge_counts["rel"] == "participants"]["count"].sum())

# ── Degree statistics ──────────────────────────────────────────────────────────
met_mean = degree_stats.loc["Metabolite",  "mean"];  met_std = degree_stats.loc["Metabolite",  "std"];  met_max  = int(degree_stats.loc["Metabolite",  "max"])
int_mean = degree_stats.loc["Interaction", "mean"];  int_std = degree_stats.loc["Interaction", "std"]
pw_mean  = degree_stats.loc["Pathway",     "mean"];  pw_max  = int(degree_stats.loc["Pathway",     "max"])

# ── Organism (taxonomy) coverage ─────────────────────────────────────────────────
# Species assignment comes from a separate companion RDF file
# (all_gpml_taxonomy_extra-*.ttl.gz) which links Protein/GeneProduct/Metabolite/DataNode
# nodes directly to NCBITaxon organisms (see CLAUDE.md "Taxonomy (species) assignment").
# The core RDF dump's own wp:organism triples are a generic Pathway-level placeholder
# (NCBITaxon_131567, "cellular organisms") and carry no species signal, so they are
# NOT used, so there is no (Pathway, organism, Organism) edge type.
# DataNode is skipped if it is not a node type in the loaded graph (see Section 9).
ORGANISM_TARGET_TYPES = [t for t in ["Protein", "GeneProduct", "Metabolite", "DataNode"]
                          if t in set(node_counts["node_type"])]
FALLBACK_TAXON_URI = "http://purl.obolibrary.org/obo/NCBITaxon_33090"  # Viridiplantae, species unspecified

org_edges = edges[edges["rel"] == "organism"]

taxa_per_node = org_edges.groupby("src")["dst"].agg(set)
fallback_nodes = set(taxa_per_node[taxa_per_node.apply(lambda s: s == {FALLBACK_TAXON_URI})].index)
multi_label_nodes = set(taxa_per_node[taxa_per_node.apply(len) > 1].index)

n_target_total = sum(safe_count(t) for t in ORGANISM_TARGET_TYPES)
n_fallback   = len(fallback_nodes)
n_multi      = len(multi_label_nodes)
n_covered    = taxa_per_node.index.nunique() - n_fallback
pct_covered  = 100 * n_covered / n_target_total
pct_fallback = 100 * n_fallback / n_target_total
pct_multi    = 100 * n_multi / n_target_total
n_organisms_real = org_edges.loc[org_edges["dst"] != FALLBACK_TAXON_URI, "dst"].nunique()

# Coverage broken down by node type
org_coverage = pd.DataFrame([
    {
        "node_type": t,
        "total":   safe_count(t),
        "covered": len(set(org_edges.loc[org_edges["src_type"] == t, "src"]) - fallback_nodes),
    }
    for t in ORGANISM_TARGET_TYPES
])
org_coverage["fallback"]    = org_coverage["total"] - org_coverage["covered"]
org_coverage["pct_covered"] = (100 * org_coverage["covered"] / org_coverage["total"]).round(1)
display(org_coverage)
org_coverage.to_csv(TABLE_DIR / "organism_coverage_by_type.tsv", sep="\t", index=False)

# Most common real (non-fallback) species by number of assigned nodes
top_taxa = (
    org_edges[org_edges["dst"] != FALLBACK_TAXON_URI]
    .assign(taxon=lambda d: d["dst"].str.rsplit("_", n=1).str[-1])
    .groupby("taxon").size().reset_index(name="n_nodes")
    .sort_values("n_nodes", ascending=False)
    .head(10)
)
display(top_taxa)

# ── External node discrepancy ──────────────────────────────────────────────────
n_ext_in_edgelist = int(degree_stats.loc["External", "count"]) if "External" in degree_stats.index else 0
n_ext_dropped     = n_ext_in_edgelist - n_external_nodes

# ── Render ─────────────────────────────────────────────────────────────────────
display(Markdown(f"""
### Graph Composition

The graph contains **{total_nodes:,} nodes** across {node_counts["node_type"].nunique()} node types and **{total_edges:,} edge instances**
across {n_edge_types} relation types.

**`Interaction` nodes dominate** ({n_interaction:,} nodes, {pct_interaction:.0f}% of the graph).
In WikiPathways, an `Interaction` represents a biochemical event (reaction, catalysis, binding) —
it is the central entity for link prediction.
The biologically meaningful entities are comparatively small:
{n_metabolite:,} Metabolites, {n_protein:,} Proteins, {n_geneproduct:,} GeneProducts,
{n_datanode:,} DataNodes, {n_pathway:,} Pathways, and {n_organism:,} Organisms (NCBITaxon species).
(`DataNode` and `External` read 0 when the cleaned graph of Section 9 is loaded, since they are not node types there.)

---

### Edge Structure

Most edge types come in **symmetric pairs** (e.g. `Interaction –[participants]→ Metabolite` and
`Metabolite –[is_part_of]→ Interaction` carry identical counts), encoding the same relationship
bidirectionally from the RDF source.

The **directional edges** (`source` / `target`, {directional_edges:,} total) are the most informative
for the link-prediction task: they encode which entities feed into a reaction and what a reaction
produces. Together with `participants` ({participants_edges:,} edges), these form the natural
supervision signal for predicting missing enzyme–reaction and metabolite–reaction links.

---

### Degree Distribution

**Metabolites are extreme hubs** (mean degree {met_mean:.1f}, std {met_std:.1f}, max {met_max:,}).
This heavy-tailed distribution reflects a small number of universal cofactors (ATP, NADH, water, CoA)
that participate in thousands of reactions. This skew requires careful handling during train/val/test
splitting to avoid data leakage through shared hub nodes.

**Interactions are relatively uniform** (mean {int_mean:.1f}, std {int_std:.1f}): most reactions
involve a small, fixed number of participants, which is biochemically expected.

**Pathways** show moderate variability (mean {pw_mean:.1f}, max {pw_max}): a few large hub pathways
(e.g. central carbon metabolism) coexist with many small specialised ones.

---

### Organism (Taxonomy) Coverage

Species information is stored in a **separate companion file**
(`all_gpml_taxonomy_extra-*.ttl.gz`), assigned directly to **Protein, GeneProduct, Metabolite,
and DataNode** nodes via `(node, organism, Organism)` edges built by
`scripts/rdf_to_typed_tables.py`. The core RDF dump's pathway-level `wp:organism` triples are a
generic `NCBITaxon_131567` ("cellular organisms") placeholder with no species signal and are not
included in the graph at all.

Out of **{n_target_total:,}** {' / '.join(ORGANISM_TARGET_TYPES)} nodes:

- **{n_covered:,} ({pct_covered:.0f}%)** have a real species assignment from the taxonomy-extra
  file, spanning **{n_organisms_real}** distinct taxa (table of top 10 above).
- **{n_fallback:,} ({pct_fallback:.0f}%)** have no entry in the taxonomy-extra file and instead
  receive a synthetic **"unknown organism"** fallback label, `NCBITaxon_33090`
  (*Viridiplantae*, i.e. green plants with species unspecified). This fallback is a deliberate
  design choice (`scripts/rdf_to_typed_tables.py`, `FALLBACK_TAXON`) so that every
  taxonomy-target node has at least one `organism` edge.
- **{n_multi:,} ({pct_multi:.1f}%)** nodes are assigned **more than one** taxon (multi-label
  organism assignment) — allowed in the graph (e.g. an enzyme conserved across several annotated
  species).

Coverage varies by node type (table above): Proteins and GeneProducts are covered far more often
than Metabolites and DataNodes, reflecting which entities WikiPathways annotates with species
directly.

**Implication for splitting**: `02_build_splits.ipynb` builds an organism-held-out ("taxa") split
using these `organism` edges. Both the fallback-labelled nodes (`NCBITaxon_33090`, "unknown
organism") and the multi-label nodes are **excluded from held-out (val/test) organism
assignment** — a node whose only species signal is "unspecified green plant", or that is shared
across several species, cannot be cleanly attributed to a single held-out organism, so its edges
always stay in train.

---

### Data Integrity Note

The degree statistics computed from `edges.tsv` reveal **{n_ext_in_edgelist:,} distinct External URIs**
referenced in the edge list, yet `heterodata.pt` contains only **{n_external_nodes} External nodes**.
The {n_ext_dropped:,} discarded entries are `is_about` links pointing to external database identifiers
(ChEBI, UniProt, etc.) that were not present in `nodes.tsv` and were therefore silently dropped by
`build_pyg.py`. If cross-database identifiers are needed as node features in future, this step of the
pipeline will need revisiting. (Section 9 acts on this directly: `External` is dropped from the
graph entirely.)
"""))

## 6. Interaction subtypes in the source RDF

`rdf_to_typed_tables.py` collapses several WikiPathways ontology classes into a single
`Interaction` node type (see `CLAUDE.md`, "Key design decisions"). The full priority-ordered
mapping (`CLASS_URI_TO_TYPE`) sends `wp:Interaction`, `gpml:Interaction`,
`wp:DirectedInteraction`, `wp:Conversion`, `wp:Catalysis`, `wp:Inhibition`, `wp:Stimulation`,
`wp:Binding`, `wp:ComplexBinding`, and `wp:TranscriptionTranslation` all to `Interaction`.

These classes have very different biological meanings (from the WikiPathways RDF ontology):

| WP class | Meaning | `source` / `target` / `participants` |
|---|---|---|
| `wp:Conversion` | A biochemical **reaction**: substrate(s) -> product(s) | Metabolites |
| `wp:Catalysis` | An **enzyme catalyses a reaction**: `source` = catalysing Protein, `target`/`participants` = the `Conversion` it controls | Protein + Interaction (the Conversion) |
| `wp:TranscriptionTranslation` | Gene expression step: GeneProduct -> Protein | GeneProduct, Protein |
| `wp:Inhibition` / `wp:Stimulation` | Regulatory edge: `source` = regulator (usually a Metabolite), `target` = the regulated `Conversion` | Metabolite + Interaction |
| `wp:Binding` / `wp:ComplexBinding` | Complex formation | Protein / DataNode |
| `wp:Interaction` / `gpml:Interaction` / `wp:DirectedInteraction` | Generic base classes carried by almost every node above | - |

To find out which of these is "the reaction" for missing-reaction prediction (CLAUDE.md:
*predict missing enzyme-reaction or metabolite-reaction links*), we recover the original
`rdf:type` assertions from `data/interim/reactions.ttl` for every node currently typed
`Interaction`, restricted to these "leaf" biological classes (excluding the generic base
classes, which almost every node also carries).

In [ ]:
INTERACTION_SUBTYPE_CACHE = TABLE_DIR / "interaction_subtypes.tsv"

WP = "http://vocabularies.wikipathways.org/wp#"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

# "Leaf" subtypes that distinguish the biological meaning of an Interaction node.
# wp:Interaction / gpml:Interaction / wp:DirectedInteraction are generic base classes
# carried by almost every Interaction node, so they're excluded here.
INTERACTION_SUBTYPE_CLASSES = {
    WP + "Conversion": "Conversion",
    WP + "Catalysis": "Catalysis",
    WP + "TranscriptionTranslation": "TranscriptionTranslation",
    WP + "Inhibition": "Inhibition",
    WP + "Stimulation": "Stimulation",
    WP + "Binding": "Binding",
    WP + "ComplexBinding": "ComplexBinding",
}

interaction_ids = set(nodes.loc[nodes["node_type"] == "Interaction", "node_id"])

if INTERACTION_SUBTYPE_CACHE.exists():
    subtypes = pd.read_csv(INTERACTION_SUBTYPE_CACHE, sep="\t")
else:
    from rdflib import Graph, URIRef
    rttl = ROOT / "data" / "interim" / "reactions.ttl"
    g = Graph()
    g.parse(rttl.as_posix(), format="turtle")
    print(f"Loaded {len(g):,} triples from {rttl}")

    rows = [
        {"node_id": str(s), "subtype": INTERACTION_SUBTYPE_CLASSES[str(o)]}
        for s, _, o in g.triples((None, URIRef(RDF_TYPE), None))
        if str(o) in INTERACTION_SUBTYPE_CLASSES and str(s) in interaction_ids
    ]
    subtypes = pd.DataFrame(rows)
    subtypes.to_csv(INTERACTION_SUBTYPE_CACHE, sep="\t", index=False)

n_interaction_total = len(interaction_ids)
n_labeled = subtypes["node_id"].nunique()
n_generic = n_interaction_total - n_labeled
print(f"Interaction nodes: {n_interaction_total:,}")
print(f"  with a recovered 'leaf' subtype:    {n_labeled:,} ({100 * n_labeled / n_interaction_total:.1f}%) "
      f"-- semantic wp:Interaction nodes")
print(f"  without one (generic/diagram-only): {n_generic:,} ({100 * n_generic / n_interaction_total:.1f}%) "
      f"-- mostly gpml:Interaction diagram connectors")

In [ ]:
subtype_counts = (
    subtypes["subtype"].value_counts()
    .rename_axis("subtype").reset_index(name="count")
)
subtype_counts["pct_of_labeled"] = (100 * subtype_counts["count"] / n_labeled).round(1)
print(subtype_counts.to_string(index=False))
subtype_counts.to_csv(TABLE_DIR / "interaction_subtype_counts.tsv", sep="\t", index=False)

fig, ax = plt.subplots()
ax.barh(subtype_counts["subtype"], subtype_counts["count"])
ax.set_xlabel("# Interaction nodes")
ax.set_title("Interaction subtypes recovered from rdf:type")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "interaction_subtypes.png")
plt.show()

In [ ]:
# What do the source/target/participants edges of each subtype actually connect to?
subtype_lookup = subtypes.drop_duplicates("node_id").set_index("node_id")["subtype"]

dir_edges = edges[
    (edges["src_type"] == "Interaction") & edges["rel"].isin(["source", "target", "participants"])
].copy()
dir_edges["subtype"] = dir_edges["src"].map(subtype_lookup).fillna("(generic, no leaf subtype)")

neighbor_breakdown = (
    dir_edges.groupby(["subtype", "rel", "dst_type"]).size()
    .reset_index(name="count")
    .sort_values(["subtype", "count"], ascending=[True, False])
)
neighbor_breakdown.to_csv(TABLE_DIR / "interaction_subtype_neighbors.tsv", sep="\t", index=False)
display(neighbor_breakdown)

In [ ]:
from IPython.display import Markdown

n_conversion = int(subtype_counts.loc[subtype_counts["subtype"] == "Conversion", "count"].iloc[0])
n_catalysis  = int(subtype_counts.loc[subtype_counts["subtype"] == "Catalysis", "count"].iloc[0])
n_transcr    = int(subtype_counts.loc[subtype_counts["subtype"] == "TranscriptionTranslation", "count"].iloc[0])
n_inhib      = int(subtype_counts.loc[subtype_counts["subtype"] == "Inhibition", "count"].iloc[0])
n_stim       = int(subtype_counts.loc[subtype_counts["subtype"] == "Stimulation", "count"].iloc[0])
n_bind       = int(subtype_counts.loc[subtype_counts["subtype"].isin(["Binding", "ComplexBinding"]), "count"].sum())

display(Markdown(f"""
### Reflection: which "Interaction" subtype is *the* reaction?

Of the {n_interaction_total:,} `Interaction` nodes, only **{n_labeled:,} ({100*n_labeled/n_interaction_total:.0f}%)**
carry a "leaf" biological subtype from `rdf:type`; the other **{n_generic:,} ({100*n_generic/n_interaction_total:.0f}%)**
are `gpml:Interaction` diagram connectors (x/y/colour/etc.) with almost no `source`/`target`/
`participants` edges -- visual layout artifacts, not biology, contributing essentially nothing to
the learning task.

Among the {n_labeled:,} biologically-typed Interaction nodes:

| Subtype | Count | % | What its `source`/`target`/`participants` connect to |
|---|---:|---:|---|
| **`Conversion`** | {n_conversion:,} | {100*n_conversion/n_labeled:.0f}% | **Metabolites only** -- substrates (`source`), products (`target`), all reactants/products (`participants`) |
| **`Catalysis`** | {n_catalysis:,} | {100*n_catalysis/n_labeled:.0f}% | `source` = catalysing **Protein**, `target`/`participants` = the **`Conversion`** it controls |
| `TranscriptionTranslation` | {n_transcr:,} | {100*n_transcr/n_labeled:.0f}% | GeneProduct -> Protein (gene expression, not metabolism) |
| `Inhibition` | {n_inhib:,} | {100*n_inhib/n_labeled:.0f}% | `source` = regulating Metabolite, `target` = the regulated `Conversion` |
| `Stimulation` | {n_stim:,} | {100*n_stim/n_labeled:.0f}% | same pattern as `Inhibition`, much rarer |
| `Binding`/`ComplexBinding` | {n_bind:,} | {100*n_bind/n_labeled:.0f}% | complex formation (Protein/DataNode), negligible |

**`Conversion` is the reaction.** It is the only subtype whose `source`/`target`/`participants`
edges connect exclusively to `Metabolite` nodes -- exactly the **metabolite-reaction links**
CLAUDE.md describes as the central prediction target. A "missing reaction" is most naturally
framed as predicting a missing `(Metabolite, participants/source/target, Conversion)` edge, or
predicting a whole new `Conversion` node and its metabolite edges.

**`Catalysis` is the enzyme-reaction link, not a reaction itself.** Every `Catalysis` node forms a
fixed pattern: `Protein --source--> Catalysis --target--> Conversion` (plus reciprocal
`participants` edges to both). This *is* the **enzyme-reaction link** from CLAUDE.md -- for an
enzyme-reaction prediction task, the natural framing is "does `Protein` X catalyse `Conversion`
Y", which can be read off (or collapsed into a direct `Protein -catalyzes-> Conversion` edge)
from these `Catalysis` nodes.

**Everything else is a different biological question.** `TranscriptionTranslation`
({n_transcr:,} nodes) is about gene expression, not metabolism. `Inhibition`/`Stimulation`
({n_inhib + n_stim:,} nodes combined) are regulatory edges where a metabolite modulates a
*Conversion* rather than participating in it -- biologically interesting, but a different
prediction target (regulation, not stoichiometry). `Binding`/`ComplexBinding` is negligible
({n_bind} nodes).

**Implication for the pipeline**: currently `Conversion`, `Catalysis`, `Inhibition`,
`Stimulation`, `TranscriptionTranslation`, `Binding`/`ComplexBinding`, and ~{n_generic:,}
diagram-only nodes are *all* the same `Interaction` node type, and their
`source`/`target`/`participants` edges are indistinguishable in `edges.tsv` / `heterodata.pt`. A
`(Metabolite, participants, Interaction)` edge could mean "is a substrate/product of this
reaction" *or* "inhibits this reaction" -- two very different biological claims a link-prediction
model currently cannot tell apart. The recovered `subtype` label (written to
`reports/tables/interaction_subtypes.tsv`) is a natural candidate for an extra `nodes.tsv`
column, so `04_link_prediction.ipynb` can:
- focus the missing-reaction target on `(Metabolite, *, Conversion)` and
  `(Protein, *, Catalysis)` / `(Catalysis, *, Conversion)` edges,
- optionally drop the {n_generic:,} diagram-only `Interaction` nodes entirely, and
- keep `Inhibition` / `Stimulation` / `TranscriptionTranslation` as separate (or excluded) edge
  types rather than mixing them into the reaction-participation signal.
"""))

## 7. Graph schema: nodes, edges, and what each means for modelling

The figure below summarises the heterogeneous graph using the refined `Interaction` subtypes
from Section 6. Each box is a node type with its node count; each arrow is a relation with its
edge count. Colours mark each type's role in the modelling plan (justified in Section 8):

- **green** = entity types to featurise (`03_*_encodings.ipynb`: gene, protein, metabolite)
- **red** = the prediction target (`Catalysis` -- enzyme/reaction assignment)
- **blue** = `Conversion`, the reaction itself: its identity is defined by its `Metabolite`
  participants (context, not the prediction target)
- **grey** = auxiliary interaction subtypes -- real biology, but a different relation than
  "catalyses"; kept as context, not as the split target
- **yellow** = `Pathway` / `Organism` / `External` metadata
- **near-white** = excluded from the learning graph

`Pathway`, `Organism`, and `External` are shown as boxes (with their node counts), but their
connecting edges are *not drawn* -- with ~96k `is_part_of` and ~54k `organism` edges fanning out
from almost every other node type, drawing all of them made the diagram unreadable (overlapping
arrows and counts). The table below the figure gives the per-node-type edge counts instead.
**These edges are not missing from the data**: they are fully present in `heterodata.pt` and
directly drive `02_build_splits.ipynb` -- the pathway-balanced split (Section 6 there) walks
`is_part_of` edges to hold out whole pathways, and the taxa split (Section 5 there) walks
`organism` edges to hold out whole species.

`is_about` edges to `External` are similarly omitted from both the figure and the table -- they
carry no reaction or taxonomy information and have no modelling role yet.

A simplified diagram showing only the nodes and edges directly involved in the `Catalysis`
link-prediction task follows after the table.

In [ ]:
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


def ncount(node_type):
    # 0 if the node type is not present (e.g. DataNode/External, removed in Section 9)
    m = node_counts.loc[node_counts["node_type"] == node_type, "count"]
    return int(m.iloc[0]) if len(m) else 0


def nb_count(subtype, rel, dst_type):
    m = neighbor_breakdown[
        (neighbor_breakdown["subtype"] == subtype)
        & (neighbor_breakdown["rel"] == rel)
        & (neighbor_breakdown["dst_type"] == dst_type)
    ]
    return int(m["count"].iloc[0]) if len(m) else 0


def curve_point(p0, p1, rad, t):
    """Point at parameter t along the arc3(rad) connection curve from p0 to p1.

    Used to place edge labels ON the drawn curve rather than on the
    straight-line midpoint, and to stagger labels of edges that share an
    endpoint (via rad/t) so they do not overlap each other or the nodes.
    """
    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    mid = (p0 + p1) / 2
    d = p1 - p0
    perp = np.array([-d[1], d[0]])
    ctrl = mid + rad * perp
    return (1 - t) ** 2 * p0 + 2 * (1 - t) * t * ctrl + t ** 2 * p1


Metabolite_count  = ncount("Metabolite")
Protein_count     = ncount("Protein")
GeneProduct_count = ncount("GeneProduct")
DataNode_count    = ncount("DataNode")
Pathway_count     = ncount("Pathway")
Organism_count    = ncount("Organism")
External_count    = ncount("External")

# label -> (count, category, (x, y))
node_info = {
    "Metabolite":               (Metabolite_count,  "feature", (0.5, 5.5)),
    "GeneProduct":              (GeneProduct_count, "feature", (0.5, 1.0)),
    "Protein":                  (Protein_count,     "feature", (9.5, 3.0)),
    "DataNode":                 (DataNode_count,    "exclude", (9.5, -1.0)),
    "Conversion":               (n_conversion,      "reaction", (3.7, 5.5)),
    "Catalysis":                (n_catalysis,       "target",   (6.8, 4.5)),
    "TranscriptionTranslation": (n_transcr,         "aux",      (3.7, 1.0)),
    "Inhibition":               (n_inhib,           "aux",      (6.8, 7.0)),
    "Stimulation":              (n_stim,            "aux",      (6.8, 8.6)),
    "Binding /\nComplexBinding": (n_bind,          "aux",      (9.5, 1.0)),
    "Pathway":                  (Pathway_count,     "meta",     (3.7, 9.7)),
    "Organism":                 (Organism_count,    "meta",     (12.2, 1.0)),
    "External":                 (External_count,    "meta",     (12.2, 4.5)),
    "Other Interaction\n(diagram-only)": (n_generic, "exclude", (0.5, -1.5)),
}

# (src, dst, edge label, count, role, rad, t)
# rad = arc curvature of the connector; t = label position along that curve
# (0 = src, 1 = dst). rad/t are tuned per edge so curves and labels that share
# an endpoint fan out instead of stacking on top of each other.
edge_list = [
    ("Conversion", "Metabolite", "participants\n(n={:,})".format(
     nb_count("Conversion", "participants", "Metabolite")), "reaction", 0.18, 0.5),
    ("Catalysis", "Protein", "source\n(n={:,})".format(
     nb_count("Catalysis", "source", "Protein")), "target", 0.18, 0.5),
    ("Catalysis", "Conversion", "target\n(n={:,})".format(
     nb_count("Catalysis", "target", "Interaction")), "target", 0.30, 0.5),
    ("Inhibition", "Metabolite", "source\n(n={:,})".format(
     nb_count("Inhibition", "source", "Metabolite")), "aux", 0.45, 0.15),
    ("Inhibition", "Conversion", "target\n(n={:,})".format(
     nb_count("Inhibition", "target", "Interaction")), "aux", -0.65, 0.4),
    ("Stimulation", "Metabolite", "source\n(n={:,})".format(
     nb_count("Stimulation", "source", "Metabolite")), "aux", 0.05, 0.5),
    ("Stimulation", "Conversion", "target\n(n={:,})".format(
     nb_count("Stimulation", "target", "Interaction")), "aux", -0.35, 0.4),
    ("TranscriptionTranslation", "GeneProduct", "source\n(n={:,})".format(
     nb_count("TranscriptionTranslation", "source", "GeneProduct")), "aux", 0.12, 0.5),
    ("TranscriptionTranslation", "Protein", "target\n(n={:,})".format(
     nb_count("TranscriptionTranslation", "target", "Protein")), "aux", 0.12, 0.5),
    ("Binding /\nComplexBinding", "Protein", "participants\n(n={:,})".format(
     nb_count("Binding", "participants", "Protein") + nb_count("ComplexBinding", "participants", "Protein")), "aux", 0.18, 0.5),
    ("Binding /\nComplexBinding", "DataNode", "participants\n(n={:,})".format(
     nb_count("Binding", "participants", "DataNode") + nb_count("ComplexBinding", "participants", "DataNode")), "aux", -0.18, 0.5),
]

color_map = {
    "feature":  "#a6d8a8",
    "target":   "#f4a3a3",
    "reaction": "#a3c9f4",
    "aux":      "#e6e6e6",
    "meta":     "#f4e3a3",
    "exclude":  "#fafafa",
}

role_styles = {
    "target":   dict(color="crimson", lw=2.5),
    "reaction": dict(color="dimgray", lw=1.0),
    "aux":      dict(color="dimgray", lw=1.0),
}

fig, ax = plt.subplots(figsize=(15, 11))
box_w, box_h = 2.2, 1.0
positions = {}
for label, (count, cat, (x, y)) in node_info.items():
    box = FancyBboxPatch((x - box_w / 2, y - box_h / 2), box_w, box_h,
                          boxstyle="round,pad=0.05", linewidth=1.3,
                          edgecolor="black", facecolor=color_map[cat], zorder=2)
    ax.add_patch(box)
    ax.text(x, y, f"{label}\n(n={count:,})", ha="center", va="center", fontsize=8.5, zorder=3)
    positions[label] = (x, y)

for src, dst, label, role, rad, t in edge_list:
    p0, p1 = positions[src], positions[dst]
    style = role_styles[role]
    arrow = FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=14,
                             color=style["color"], lw=style["lw"],
                             connectionstyle=f"arc3,rad={rad}",
                             shrinkA=28, shrinkB=28, zorder=1)
    ax.add_patch(arrow)
    lx, ly = curve_point(p0, p1, rad, t)
    ax.text(lx, ly, label, fontsize=7, ha="center", va="center",
            color=style["color"], bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.2), zorder=4)

legend_items = [
    ("To featurise (03_*_encodings)", color_map["feature"]),
    ("Prediction target (Catalysis)", color_map["target"]),
    ("Reaction identity / context", color_map["reaction"]),
    ("Auxiliary context (not target)", color_map["aux"]),
    ("Pathway / Organism / External metadata", color_map["meta"]),
    ("Excluded from learning graph", color_map["exclude"]),
]
legend_patches = [mpatches.Patch(facecolor=c, edgecolor="black", label=l) for l, c in legend_items]
ax.legend(handles=legend_patches, loc="lower center", bbox_to_anchor=(0.5, -0.08),
          ncol=3, fontsize=8.5, framealpha=0.95)

ax.set_xlim(-1.7, 13.7)
ax.set_ylim(-3, 10.7)
ax.axis("off")
ax.set_title("PlantMetWiki heterogeneous graph schema (Interaction split by recovered subtype)")
plt.tight_layout()
plt.savefig(FIG_DIR / "graph_schema.png", dpi=150, bbox_inches="tight")
plt.show()

**Figure -- PlantMetWiki heterogeneous graph schema.** Each box is a node type
(for `Interaction`, a recovered subtype from Section 6) labelled with its node count `n`. Arrows
are the "reaction structure" edges (`source` / `target` / `participants`) with their edge
counts; red arrows mark the `Catalysis` edges that form the link-prediction target
`(Protein, catalyzes, Interaction)` (Section 8). `Pathway`, `Organism`, and `External` are shown
as boxes only -- their `is_part_of` / `organism` / `is_about` edges are omitted from the figure
for readability and summarised in the table below instead. Box colours indicate each type's role
in the modelling plan: green = entities to featurise, red = prediction target, blue = reaction
identity/context, grey = auxiliary context (kept but not the split target), yellow =
pathway/organism/external metadata, near-white = excluded from the learning graph.

In [ ]:
from IPython.display import Markdown, display

pathway_edges = edges[(edges["rel"] == "is_part_of") & (edges["dst_type"] == "Pathway")].copy()

int_pw = pathway_edges[pathway_edges["src_type"] == "Interaction"].copy()
int_pw["subtype"] = int_pw["src"].map(subtype_lookup).fillna("(generic, no leaf subtype)")
int_pw_counts = int_pw.groupby("subtype").size()

entity_pw_counts = (
    pathway_edges[pathway_edges["src_type"] != "Interaction"]
    .groupby("src_type").size()
)

org_edges = edges[(edges["rel"] == "organism") & (edges["dst_type"] == "Organism")]
org_counts = org_edges.groupby("src_type").size()

# (table row label, subtype key(s) into int_pw_counts) for the Interaction rows
SUBTYPE_ROWS = [
    ("Interaction: Conversion", "Conversion"),
    ("Interaction: Catalysis", "Catalysis"),
    ("Interaction: TranscriptionTranslation", "TranscriptionTranslation"),
    ("Interaction: Inhibition", "Inhibition"),
    ("Interaction: Stimulation", "Stimulation"),
    ("Interaction: Binding / ComplexBinding", ["Binding", "ComplexBinding"]),
    ("Interaction: generic (no leaf subtype)", "(generic, no leaf subtype)"),
]

pw_rows = []
for row_label, key in SUBTYPE_ROWS:
    keys = key if isinstance(key, list) else [key]
    pw_rows.append((row_label, sum(int(int_pw_counts.get(k, 0)) for k in keys)))
for src_type in ["Metabolite", "Protein", "GeneProduct", "DataNode"]:
    pw_rows.append((src_type, int(entity_pw_counts.get(src_type, 0))))
pw_total = sum(c for _, c in pw_rows)

org_rows = [(src_type, int(org_counts.get(src_type, 0)))
            for src_type in ["Protein", "GeneProduct", "Metabolite", "DataNode"]]
org_total = sum(c for _, c in org_rows)


def to_table(rows, total, col2):
    lines = [f"| Source node type | {col2} |", "|---|---:|"]
    lines += [f"| {name} | {cnt:,} |" for name, cnt in rows]
    lines.append(f"| **Total** | **{total:,}** |")
    return "\n".join(lines)


display(Markdown(
    "**`is_part_of` edges to `Pathway`** (used by the pathway-balanced split, "
    "`02_build_splits.ipynb` Section 6):\n\n"
    + to_table(pw_rows, pw_total, "Edges to Pathway")
    + "\n\n**`organism` edges to `Organism`** (used by the taxa split, "
    "`02_build_splits.ipynb` Section 5):\n\n"
    + to_table(org_rows, org_total, "Edges to Organism")
))

In [ ]:
core_node_info = {
    "Metabolite": (Metabolite_count, "feature",  (0.5, 3.0)),
    "Protein":    (Protein_count,    "feature",  (8.5, 3.0)),
    "Conversion": (n_conversion,     "reaction", (3.0, 5.0)),
    "Catalysis":  (n_catalysis,      "target",   (6.0, 5.0)),
}

core_edge_list = [
    ("Conversion", "Metabolite", "participants\n(substrate/product)",
     nb_count("Conversion", "participants", "Metabolite"), "reaction", 0.12, 0.5),
    ("Catalysis", "Protein", "source\n(enzyme)",
     nb_count("Catalysis", "source", "Protein"), "target", 0.12, 0.5),
    ("Catalysis", "Conversion", "target\n(reaction catalysed)",
     nb_count("Catalysis", "target", "Interaction"), "target", -0.6, 0.5),
]

fig, ax = plt.subplots(figsize=(8, 6))
positions = {}
for label, (count, cat, (x, y)) in core_node_info.items():
    box = FancyBboxPatch((x - box_w / 2, y - box_h / 2), box_w, box_h,
                          boxstyle="round,pad=0.05", linewidth=1.3,
                          edgecolor="black", facecolor=color_map[cat], zorder=2)
    ax.add_patch(box)
    ax.text(x, y, f"{label}\n(n={count:,})", ha="center", va="center", fontsize=9, zorder=3)
    positions[label] = (x, y)

for src, dst, label, count, role, rad, t in core_edge_list:
    p0, p1 = positions[src], positions[dst]
    style = role_styles[role]
    arrow = FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=14,
                             color=style["color"], lw=style["lw"],
                             connectionstyle=f"arc3,rad={rad}",
                             shrinkA=30, shrinkB=30, zorder=1)
    ax.add_patch(arrow)
    lx, ly = curve_point(p0, p1, rad, t)
    ax.text(lx, ly, f"{label}\n(n={count:,})", fontsize=8, ha="center", va="center",
            color=style["color"], bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1.5), zorder=4)

legend_patches_core = [
    mpatches.Patch(facecolor=color_map["feature"], edgecolor="black", label="To featurise (03_*_encodings)"),
    mpatches.Patch(facecolor=color_map["target"], edgecolor="black", label="Prediction target (Catalysis)"),
    mpatches.Patch(facecolor=color_map["reaction"], edgecolor="black", label="Reaction identity / context"),
]
ax.legend(handles=legend_patches_core, loc="lower center", bbox_to_anchor=(0.5, -0.18),
          ncol=3, fontsize=9, framealpha=0.95)

ax.set_xlim(-1.5, 10.5)
ax.set_ylim(1.8, 7.3)
ax.axis("off")
ax.set_title("Core schema for the Catalysis link-prediction task")
plt.tight_layout()
plt.savefig(FIG_DIR / "graph_schema_core.png", dpi=150, bbox_inches="tight")
plt.show()

**Figure -- Core schema for the `Catalysis` link-prediction task.** The minimal
subgraph directly relevant to the prediction target: `Catalysis` interactions link a `Protein`
(the enzyme, `source`) to a `Conversion` (the reaction, `target`), and `Conversion` interactions
link to their `Metabolite` participants (substrates/products), which give the reaction its
chemical identity. This is the subgraph that `(Protein, catalyzes, Interaction)` link prediction
(Section 8) operates over; `GeneProduct`, `DataNode`, the other `Interaction` subtypes, `Pathway`,
`Organism`, and `External` (all shown in the full schema above) provide additional context but
are not part of this core loop.

## 8. Prediction task: assigning enzymes to reactions

**Task framing (Conversion → Protein direction).** The task is to predict missing `(Protein, catalyzes, Conversion)` edges: *given a biochemical reaction, which protein catalyses it?*

The framing follows from the annotation asymmetry in the data:

| | Count | Share |
|--|--|--|
| Conversion (reaction) nodes | 84,470 | 100% |
| Reactions with ≥1 known catalyst | 4,572 | 5.4% |
| Reactions with no known catalyst (orphan) | 79,898 | 94.6% |
| Protein nodes | 13,682 | 100% |
| Proteins annotated as any enzyme | 3,456 | 25.3% |
| Proteins with no catalysis annotation | 10,226 | 74.7% |
| (Protein, catalyzes, Conversion) positive edges | 11,980 | |

The database encodes the reactions (substrates, products, EC class) far more completely than it encodes which genes catalyse them, so the model is meant to fill the enzyme-assignment gap, especially for under-characterised organisms.

The reverse direction (given a protein, which reactions does it catalyse?) is protein function annotation. It is a valid question but not the bottleneck: the reactions and pathways are relatively well documented, while the enzyme-gene assignment within each pathway is what is missing. A biochemist reconstructing a secondary metabolic pathway in a new plant species starts from known reactions, since the chemistry is conserved, and needs to find which of the plant's genes encodes each enzyme.

**Annotation sparsity.** Among the 4,572 reactions with a known catalyst, 51.6% have exactly one, 39.5% have two to five (isoenzymes or multi-species assignments) and 8.9% have more than five (well-studied core reactions such as glycolysis). The median is one catalyst per reaction and the mean 2.62.

**Cross-species generalisation.** The graph covers 424 organisms. The taxa split (built in `03_embeddings_fulldata.ipynb` §7.8) holds out 11 val and 69 test organisms entirely, so their proteins are never seen during training. The model has to use protein sequence (ESM-C, 960-dim, available for 61.7% of proteins) and reaction chemistry to transfer enzyme-reaction knowledge from characterised to novel organisms. The 38.3% of proteins without an embedding cannot be ranked meaningfully, which caps what any model can predict.

**Operationalisation.** Link prediction on the derived `(Protein, catalyzes, Conversion)` edge type: 11,980 positive pairs after the reified `Catalysis` nodes are collapsed (§8.1). The ranking pool for evaluation is the 8,445 proteins with an ESM embedding (`--embedded-only-ranking`); proteins without one are excluded because their all-zero vectors are indistinguishable. The primary metric is **P-H@K** (Protein Hits at K): for each val/test Conversion, is the true catalyst among the top K of the 8,445 ranked proteins?

```
Random P-H@50 baseline = 50 / 8,445 ≈ 0.59%
```

This matches how WikiPathways pathways are authored. A `Conversion` is drawn first, with its substrate and product `Metabolite`s defining the reaction, and is then annotated with zero or more `Catalysis` records saying which enzyme performs it. Reconstructing a pathway means recovering the missing enzyme assignments from the known `Conversion` nodes and the candidate `Protein` nodes.

In [ ]:
catalysis_ids = set(subtypes.loc[subtypes["subtype"] == "Catalysis", "node_id"])

cat_targets = edges[(edges["src"].isin(catalysis_ids)) & (edges["rel"] == "target")]
cat_sources = edges[(edges["src"].isin(catalysis_ids)) & (edges["rel"] == "source")]

n_conv_with_catalysis = cat_targets["dst"].nunique()
pct_conv_with_cat = 100 * n_conv_with_catalysis / n_conversion

n_enzymes = cat_sources["dst"].nunique()
pct_proteins_as_enzyme = 100 * n_enzymes / Protein_count

n_dn_connected = len(set(edges.loc[(edges["rel"] != "organism") & (edges["src_type"] == "DataNode"), "src"])
                      | set(edges.loc[(edges["rel"] != "organism") & (edges["dst_type"] == "DataNode"), "dst"]))
n_dn_isolated = DataNode_count - n_dn_connected

print(f"Conversions with >=1 Catalysis edge: {n_conv_with_catalysis:,} / {n_conversion:,} ({pct_conv_with_cat:.1f}%)")
print(f"  -> Conversions with NO known catalysing enzyme: {n_conversion - n_conv_with_catalysis:,} ({100 - pct_conv_with_cat:.1f}%)")
print(f"Distinct Proteins acting as a catalysing enzyme: {n_enzymes:,} / {Protein_count:,} ({pct_proteins_as_enzyme:.1f}%)")
if DataNode_count:
    print(f"DataNodes with only organism edges (or none): {n_dn_isolated:,} / {DataNode_count:,} "
          f"({100 * n_dn_isolated / DataNode_count:.1f}%)")
else:
    print("DataNode is not a node type in the loaded graph (it is removed in Section 9).")

In [ ]:
from IPython.display import Markdown

dn_row = (
    f"| **`DataNode`** (n={DataNode_count:,}) | **Drop**, or at least exclude from message passing for this task. | "
    f"**{n_dn_isolated:,} / {DataNode_count:,} ({100 * n_dn_isolated / DataNode_count:.1f}%)** have *only* `organism` "
    f"edges -- essentially isolated from the reaction graph. |"
    if DataNode_count else
    "| **`DataNode`** | *(removed in Section 9)* | -- |"
)

display(Markdown(f"""
### Diagnosis: what to do with each edge type

| Edge type | Treatment | Reasoning |
|---|---|---|
| **`Catalysis`** (`Protein --source--> Catalysis --target--> Conversion`, n={n_catalysis:,}) | **Prediction target.** Collapse to a derived `(Protein, catalyzes, Conversion)` edge and split *that* into train/val/test. | The enzyme-reaction assignment. Only **{pct_conv_with_cat:.0f}%** of `Conversion`s currently have one -- the other **{100 - pct_conv_with_cat:.0f}%** ({n_conversion - n_conv_with_catalysis:,} reactions) are exactly the "missing reactions" to reconstruct. |
| **`Conversion` participants** (`Metabolite <-> Conversion`, n={nb_count("Conversion","participants","Metabolite"):,}) | **Context / input graph, always kept.** Used for message passing and to give each `Conversion` a "reaction identity" (its substrate/product set). | A `Conversion`'s metabolites define *what reaction it is*. Treating these as a second prediction target would conflate "which reaction is this" with "which enzyme does it" -- a different question. Could become a *separate* future task ("predict missing reaction participants"), but not this one. |
| **`TranscriptionTranslation`** (`GeneProduct --source--> ... --target--> Protein`, n={n_transcr:,}) | **Keep as an auxiliary `encodes` context edge** (`GeneProduct -> Protein`). | `Catalysis` only ever points at a `Protein`, never a `GeneProduct` (Section 6). This edge is the only link from `GeneProduct` into the reaction graph, so it lets gene-level features (`03_gene_encodings.ipynb`) reach the enzyme side. |
| **`Inhibition` / `Stimulation`** (`Metabolite --source--> ... --target--> Conversion`, n={n_inhib + n_stim:,}) | **Keep as a separate `regulates` context edge type, excluded from the target.** | Biologically real (allosteric regulation), but "this metabolite regulates this reaction" != "this metabolite is a substrate/product". Now that they're distinguishable via `subtype`, don't let them collapse into `Conversion`'s `participants` edges. |
| **`Binding` / `ComplexBinding`** (n={n_bind}) | **Drop.** | Negligible ({n_bind} nodes), adds a `Complex` concept for no real benefit at this stage. |
| **Generic `Interaction`** (diagram-only, n={n_generic:,}) | **Drop entirely** from `nodes.tsv` / `heterodata.pt`. | {100 * n_generic / n_interaction_total:.0f}% of "Interaction" nodes, almost no `source`/`target`/`participants` edges -- pure GPML layout artifacts. |
{dn_row}

### Concrete change for `02_build_splits.ipynb`

`TARGET_EDGE` is currently `("Protein", "is_part_of", "Interaction")` -- **{20645:,} edges that mix
three different subtypes**: `Catalysis` ({n_catalysis:,}, the enzyme-reaction link we actually
want), `TranscriptionTranslation` ({n_transcr:,}, gene expression), and `Stimulation` (a handful).
Splitting on this edge would make a large share of the "positive" examples mean "this protein is the product of this gene" rather than "this protein catalyses this reaction", which is a different task.

**Target edge**: `(Protein, is_part_of, Interaction)` is filtered to the `Interaction` endpoints whose recovered `subtype == "Catalysis"` (n={n_catalysis:,} edges, covering {pct_conv_with_cat:.0f}% of `Conversion` nodes), which is the "reconstruct which enzyme catalyses which reaction" task. The `interaction_subtype` label is carried into `nodes.tsv` by `rdf_to_typed_tables.py`, so the filter needs no re-parsing of the RDF. Sections 8.1 and 9 collapse these edges into the direct `catalyzes` edge.
"""))

### 8.1 Collapsing `Catalysis` into a direct edge

A `Catalysis` node could in principle attach to several Proteins (a multi-subunit enzyme complex) or several Conversions (one catalytic event enabling several reactions), in which case a single dyadic edge would lose information. The cells below check the exact cardinality on the real data, using the `cat_sources` and `cat_targets` edge slices from Section 8.

In [ ]:
cat_out_target = cat_targets.groupby("src").size()
cat_out_source = cat_sources.groupby("src").size()
print(f"Catalysis nodes with exactly 1 'target' edge (-> Conversion): {(cat_out_target == 1).sum():,} / {n_catalysis:,}")
print(f"Catalysis nodes with exactly 1 'source' edge (-> Protein):    {(cat_out_source == 1).sum():,} / {n_catalysis:,}")
print(f"Catalysis nodes with >1 of either (would block a clean collapse): "
      f"{int((cat_out_target > 1).sum() + (cat_out_source > 1).sum())}")
print()

prot_cat_conv = (
    cat_sources[["src", "dst"]].rename(columns={"dst": "protein"})
    .merge(cat_targets[["src", "dst"]].rename(columns={"dst": "conversion"}), on="src")
)
n_distinct_pairs = prot_cat_conv[["protein", "conversion"]].drop_duplicates().shape[0]
n_distinct_conv = prot_cat_conv["conversion"].nunique()

if n_distinct_conv == 0:
    print("Catalysis nodes are not present in the loaded graph (Section 9 collapses them into "
          "direct edges), so there is nothing to check here. The cardinality check applies to the "
          "graph before Section 9's cleaning.")
    fan_in = pd.Series(dtype=int)
else:
    print(f"(Protein, Conversion) pairs if collapsed directly: {n_distinct_pairs:,} distinct pairs "
          f"from {len(prot_cat_conv):,} Catalysis instances")
    print(f"  -> only {len(prot_cat_conv) - n_distinct_pairs} Catalysis instances are exact duplicates "
          f"(same protein, same conversion, via a different Catalysis node) -- negligible data loss")
    print(f"Distinct Conversions touched: {n_distinct_conv:,} "
          f"(vs {n_catalysis:,} Catalysis instances today -- the H@K ranking pool would shrink "
          f"~{n_catalysis / n_distinct_conv:.1f}x, to a biologically meaningful pool of *reactions* "
          f"rather than *enzyme-reaction instance records*)")

    fan_in = prot_cat_conv.groupby("conversion").size()
    print()
    print("Catalysis instances per Conversion (fan-in -- different enzymes/isozymes/organisms catalysing the same reaction):")
    print(fan_in.value_counts().sort_index().rename("n_conversions").to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
fan_in.value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C78A8", edgecolor="black")
ax.set_xlabel("# Catalysis instances pointing at the same Conversion")
ax.set_ylabel("# Conversion nodes")
ax.set_title("Fan-in: how many Catalysis records collapse onto one reaction")
plt.tight_layout()
plt.savefig(FIG_DIR / "catalysis_conversion_fanin.png", dpi=150)
plt.show()
print(f"Saved -> {FIG_DIR}/catalysis_conversion_fanin.png")

**Result: `Catalysis` is a reified edge.** Every Catalysis node has exactly one `source` (Protein) and one `target` (Conversion), with no multi-subunit or multi-reaction fan-out. Chaining `Protein -source-> Catalysis -target-> Conversion` into a single `(Protein, catalyzes, Conversion)` edge therefore loses essentially nothing: 12,359 Catalysis instances give 11,980 distinct positive `(Protein, Conversion)` pairs (a few exact-duplicate instances are merged).

**Candidate pool.** After the collapse, the H@K candidate pool is every Conversion-subtype node (19,927), including the 15,355 Conversions with no known catalyst. This ranks against the full universe of known reactions rather than a pre-filtered set that already has an answer.

**Message passing.** `Conversion` nodes connect directly to their substrate and product `Metabolite`s (`source`, `target`, `participants`). Through a reified `Catalysis` node, reaction chemistry sits three hops from `Protein`:

```
Protein --source--> Catalysis (1 hop) --target--> Conversion (2 hops) --...--> Metabolite (3 hops)
```

The models in `04_link_prediction.ipynb` use two `HeteroConv` layers, which would leave the chemistry outside the receptive field. The direct edge puts it inside:

```
Protein --catalyzes--> Conversion (1 hop) --...--> Metabolite (2 hops)
```

The direct `(Protein, catalyzes, Interaction)` edge is persisted in `edges.tsv` and `heterodata.pt`, and `load_taxa_splits()` in `training/fulldata_baseline_common.py` reads it from there.


## 9. Graph cleaning: removing noise node types, collapsing reified edges

Four structural changes are applied to `data/processed/nodes.tsv` and `edges.tsv` (and so to `heterodata.pt`) once, as a graph-build step rather than at training time:

1. **`DataNode` (42,493 nodes).** These are generic, pathway-local diagram elements that are never deduplicated across pathways (Section 6). Each is classified by its local-identifier pattern as `GeneProduct`, `Protein`, `Metabolite` or unclassifiable. A classified entry whose identifier exactly matches an existing canonical node of the same type is merged into it (edges are redirected and the duplicate dropped, which recovers a real precomputed embedding); the others are retyped in place and keep their `node_id`. Unclassifiable entries are dropped.
2. **`External` (70 nodes).** Untyped leftovers (an artefact of the `is_about` labelling in Section 6), dropped.
3. **`Inhibition`, `Stimulation` and `ComplexBinding`** Interaction subtypes (3,486 + 865 + 70 = 4,421 nodes). Dropped rather than collapsed, because no single dyadic edge represents them without losing information.
4. **`Catalysis` (12,359 nodes) and `TranscriptionTranslation` (8,280 nodes).** Both are reified relations. They become persisted edges, `(Protein, catalyzes, Interaction)` and `(GeneProduct, encodes, Protein)`, each with a reverse edge, and are then removed as nodes. Section 8.1 shows that the `Catalysis` collapse is lossless. `TranscriptionTranslation` nodes likewise have exactly one `GeneProduct` source and one `Protein` target.

`Conversion`-subtype and blank-subtype Interaction nodes are unchanged. The loaded `nodes`, `edges` and `data` objects are the cleaned graph.
